In [73]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental SheelTool

  Using cached duckduckgo_search-8.1.1-py3-none-any.whl.metadata (16 kB)
ERROR: Could not find a version that satisfies the requirement SheelTool (from versions: none)
ERROR: No matching distribution found for SheelTool


In [74]:
pip install requests==2.32.4


In [75]:
!pip uninstall -y duckduckgo-search


In [76]:
!pip install -U ddgs


 ##  Built in tools
1. Duck Duck search

In [77]:
from langchain_community.tools import DuckDuckGoSearchRun
# ab hum aik obj bannae ga
search_tool=DuckDuckGoSearchRun()
result = search_tool.invoke(' ai news')
print(result)

AI News is an umbrella term for the continuously updated informational ecosystem reporting on artificial intelligence developments, including research results, model releases, product launches, corporate strategy, regulation, labor impacts, security… 6 days ago · AI News delivers the latest updates in artificial intelligence, machine learning, deep learning, enterprise AI, and emerging tech worldwide. Read full articles, watch videos, browse thousands of titles and more on the "Artificial intelligence" topic with Google News. 8 hours ago · Today's most important AI news, ranked and explained. Track artificial intelligence models, agents, funding, policy and research with updates throughout the day. 8 hours ago · Explore the latest artificial intelligence news with Reuters - from AI breakthroughs and technology trends to regulation, ethics, business and global impact.


## 2- SHELL TOOL

In [78]:
# shell tool sa commansd line par ja ke command excute kar sakte hoo
from langchain_community.tools import ShellTool
shell_tool = ShellTool()
result = shell_tool.invoke("whoami")
print(result)

Executing command:
 whoami
root



/usr/local/lib/python3.13/dist-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


## CUSTOM TOOLS

In [79]:
from langchain_core.tools import Tool

In [80]:
# sab sa pehle hum aik function bannae ga
def multiply(a,b):
  """multiplt two number"""
  return a*b

In [81]:
# step2 add hint type
def multiply(a:int,b:int) -> int:
  """multiplt two number"""
  return a*b

In [82]:
from langchain_core.tools import tool
# is ma hum tool decorator laga dete hai jis sa aik special function hota hain
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

In [83]:
result=multiply.invoke({'a':66,'b':78})
print(result)

5148


In [84]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [85]:
# ager ap na iska json schema dekhna
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


## 2 Structured Tool

In [86]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [87]:
class MultiplyInput(BaseModel):
  a:int=Field(required=True, description='the first number to  add')
  b:int=Field(required=True, description='the second number to  add')

/tmp/ipykernel_514/133662256.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  a:int=Field(required=True, description='the first number to  add')
/tmp/ipykernel_514/133662256.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  b:int=Field(required=True, description='the second number to  add')


In [88]:
def multiply_func(a:int,b:int) -> int:
 return a*b

In [89]:
# ab hum aik structured function bannae aur sath hi args schema abnna rahe ho
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [90]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'the first number to  add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'the second number to  add', 'title': 'B', 'type': 'integer'}}


## using Basetool class

In [91]:
from langchain_core.tools import BaseTool
from typing import Type

In [92]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_514/908171234.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_514/908171234.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  b: int = Field(required=True, description="The second number to add")


In [93]:
# ab humari multiply tool class base tool class ko inherit kar rahi hai aur sath hi basetool class ki child class hai
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [94]:
multiply_tool = MultiplyTool()

In [95]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

9
multiply
Multiply two numbers


## TOOLkIT

In [96]:
# ager ap multiple tool banna rahe ho aur unko apas ma connect kar rahe is ko tool kit kehte hai
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b
@tool
def divide(a: int, b: int) -> int:
    """divide two numbers"""
    return a / b


In [97]:
# hume jo toolkit bannai hai uska liye hum aik class banate hai
class MathToolkit:
  # yaha par apko aik gettool name ka method define karna hai aur ap return karo jo ap chate hoo
    def get_tools(self):
        return [add, multiply,divide]


In [98]:
# ap aik obj bannao ag aur jaise toolkit .get karo ga ap apne saare tools mil jae ga
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)

add => Add two numbers
multiply => Multiply two numbers
divide => divide two numbers
